In [21]:
"""
===========================================================
WEATHER ETL PIPELINE
AnalystLab Africa - Week 7 Project

Description:
    Extracts live weather data from the OpenWeather API,
    transforms it into a clean dataset, stores it in CSV,
    Excel, and SQLite, performs basic analysis, and
    generates charts.

SETUP (do this before running):
    1. Install dependencies:
       pip install pandas requests matplotlib openpyxl python-dotenv

    2. Create a file named ".env" in the same folder as this script
       (NOT inside this .py file) containing one line:
       OPENWEATHER_API_KEY=your_real_key_here

    3. Run:
       python weather_etl_pipeline.py

    All outputs are written to a "data" folder created next to this script.
===========================================================
"""

import os
import sqlite3
import logging
from datetime import datetime

import requests
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

# ===========================================================
# CONFIGURATION
# ===========================================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)

load_dotenv()

API_KEY = os.getenv("OPENWEATHER_API_KEY")

# Notebook fallback: if no .env file was found, prompt for the key instead.
# getpass hides the input as you type and never writes it to disk/notebook.
if not API_KEY:
    try:
        from getpass import getpass
        API_KEY = getpass("Enter your OpenWeather API key: ").strip() or None
    except Exception:
        pass
BASE_URL = "https://api.openweathermap.org/data/2.5/weather"

CITIES = ["Lagos", "Abuja", "Port Harcourt"]

OUTPUT_FOLDER = "data"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# Maps raw OpenWeather JSON fields to clean, analysis-ready column names.
COLUMN_RENAME_MAP = {
    "city_name": "City",
    "country": "Country",
    "temp": "Temperature_C",
    "feels_like": "FeelsLike_C",
    "humidity": "Humidity_pct",
    "pressure": "Pressure_hPa",
    "weather_main": "WeatherCondition",
    "weather_description": "Description",
    "wind_speed": "WindSpeed_ms",
    "cloudiness": "Cloudiness_pct",
    "lat": "Latitude",
    "lon": "Longitude",
    "recorded_at": "Timestamp",
}


# ===========================================================
# EXTRACT
# ===========================================================

def extract_weather(city: str) -> dict:
    """Call the OpenWeather API and return the raw fields for one city."""
    params = {"q": city, "appid": API_KEY, "units": "metric"}
    response = requests.get(BASE_URL, params=params, timeout=30)
    response.raise_for_status()
    data = response.json()

    return {
        "city_name": data.get("name", city),
        "country": data.get("sys", {}).get("country"),
        "temp": data.get("main", {}).get("temp"),
        "feels_like": data.get("main", {}).get("feels_like"),
        "humidity": data.get("main", {}).get("humidity"),
        "pressure": data.get("main", {}).get("pressure"),
        "weather_main": data.get("weather", [{}])[0].get("main"),
        "weather_description": data.get("weather", [{}])[0].get("description"),
        "wind_speed": data.get("wind", {}).get("speed"),
        "cloudiness": data.get("clouds", {}).get("all"),
        "lat": data.get("coord", {}).get("lat"),
        "lon": data.get("coord", {}).get("lon"),
        "recorded_at": datetime.now(),
    }


def extract_all(cities: list) -> list:
    """Extract weather for every city, logging and skipping failures."""
    records = []
    for city in cities:
        try:
            logging.info(f"Extracting {city}")
            records.append(extract_weather(city))
        except requests.exceptions.RequestException as e:
            logging.error(f"{city} failed (network/API error): {e}")
        except (KeyError, IndexError) as e:
            logging.error(f"{city} failed (unexpected response shape): {e}")
    return records


# ===========================================================
# TRANSFORM
# ===========================================================

def transform_data(records: list) -> pd.DataFrame:
    """Clean the raw records: rename columns, fix dtypes, sort, dedupe."""
    if not records:
        raise ValueError("No records to transform - all API calls failed.")

    df = pd.DataFrame(records)
    df = df.rename(columns=COLUMN_RENAME_MAP)

    numeric_columns = [
        "Temperature_C", "FeelsLike_C", "Humidity_pct", "Pressure_hPa",
        "WindSpeed_ms", "Cloudiness_pct", "Latitude", "Longitude",
    ]
    for col in numeric_columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["Timestamp"] = pd.to_datetime(df["Timestamp"])
    df = df.dropna(subset=["City", "Temperature_C"])
    df = df.sort_values("City").reset_index(drop=True)

    return df


# ===========================================================
# LOAD
# ===========================================================

def save_csv(df: pd.DataFrame) -> None:
    path = os.path.join(OUTPUT_FOLDER, "weather_data.csv")
    df.to_csv(path, index=False)
    logging.info(f"CSV saved to {path}")


def save_excel(df: pd.DataFrame) -> None:
    path = os.path.join(OUTPUT_FOLDER, "weather_data.xlsx")
    df.to_excel(path, index=False)
    logging.info(f"Excel saved to {path}")


def save_database(df: pd.DataFrame) -> None:
    path = os.path.join(OUTPUT_FOLDER, "weather.db")
    with sqlite3.connect(path) as conn:
        df.to_sql("weather_data", conn, if_exists="replace", index=False)
    logging.info(f"SQLite database saved to {path}")


# ===========================================================
# ANALYSIS
# ===========================================================

def analyze(df: pd.DataFrame) -> str:
    """Print and return a plain-text summary of key findings."""
    hottest = df.loc[df["Temperature_C"].idxmax()]
    most_humid = df.loc[df["Humidity_pct"].idxmax()]
    windiest = df.loc[df["WindSpeed_ms"].idxmax()]

    lines = [
        "================ WEATHER ANALYSIS ================",
        "",
        f"Average temperature: {df['Temperature_C'].mean():.2f} °C",
        f"Average humidity:    {df['Humidity_pct'].mean():.2f} %",
        f"Average wind speed:  {df['WindSpeed_ms'].mean():.2f} m/s",
        "",
        f"Hottest city:  {hottest['City']} ({hottest['Temperature_C']} °C)",
        f"Most humid:    {most_humid['City']} ({most_humid['Humidity_pct']} %)",
        f"Windiest city: {windiest['City']} ({windiest['WindSpeed_ms']} m/s)",
        "",
        "Weather conditions observed:",
        df["WeatherCondition"].value_counts().to_string(),
    ]
    summary = "\n".join(lines)
    print("\n" + summary + "\n")

    path = os.path.join(OUTPUT_FOLDER, "analysis_summary.txt")
    with open(path, "w") as f:
        f.write(summary)
    logging.info(f"Analysis summary saved to {path}")

    return summary


# ===========================================================
# VISUALIZATIONS
# ===========================================================

def _bar_chart(df, y_col, title, ylabel, filename):
    plt.figure(figsize=(8, 5))
    plt.bar(df["City"], df[y_col])
    plt.title(title)
    plt.xlabel("City")
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_FOLDER, filename))
    plt.close()


def create_charts(df: pd.DataFrame) -> None:
    _bar_chart(df, "Temperature_C", "Temperature Comparison", "Temperature (°C)", "temperature_chart.png")
    _bar_chart(df, "Humidity_pct", "Humidity Comparison", "Humidity (%)", "humidity_chart.png")
    _bar_chart(df, "WindSpeed_ms", "Wind Speed Comparison", "Wind Speed (m/s)", "windspeed_chart.png")

    weather_counts = df["WeatherCondition"].value_counts()
    plt.figure(figsize=(7, 5))
    plt.bar(weather_counts.index, weather_counts.values)
    plt.title("Weather Condition Count")
    plt.xlabel("Weather")
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_FOLDER, "weather_condition_chart.png"))
    plt.close()

    logging.info("Charts created.")


# ===========================================================
# MAIN
# ===========================================================

def main():
    logging.info("Starting ETL Pipeline")

    if not API_KEY:
        raise ValueError(
            "OPENWEATHER_API_KEY not found. Create a .env file next to "
            "this script containing: OPENWEATHER_API_KEY=your_real_key_here"
        )

    raw_records = extract_all(CITIES)
    df = transform_data(raw_records)

    save_csv(df)
    save_excel(df)
    save_database(df)

    analyze(df)
    create_charts(df)

    logging.info("ETL Pipeline Completed Successfully")


if __name__ == "__main__":
    main()


================ WEATHER ANALYSIS ================

Average temperature: 24.15 °C
Average humidity:    90.33 %
Average wind speed:  2.62 m/s

Hottest city:  Port Harcourt (24.46 °C)
Most humid:    Abuja (91 %)
Windiest city: Port Harcourt (3.26 m/s)

Weather conditions observed:
WeatherCondition
Rain    3



In [22]:
import os
print(os.getcwd())
print(os.listdir('.'))

/content
['.config', 'data', 'sample_data']


In [23]:
import shutil
from google.colab import files

shutil.make_archive('weather_etl_output', 'zip', 'data')
files.download('weather_etl_output.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [25]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copytree('data', '/content/drive/MyDrive/Colab Notebooks/weather_etl_data')

Mounted at /content/drive


'/content/drive/MyDrive/Colab Notebooks/weather_etl_data'